[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/02_dataset_prep.ipynb)

# Notebook 2 — Dataset Preparation

Download StepGame, split into train/eval, format for fine-tuning.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/spatialft.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spatialft/spatialft.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.colab_utils import ensure_notebook_requirements, prepare_notebook, publish_artifacts

REPO, PATHS = prepare_notebook(REPO)
print(f'Repo ready at {REPO}')

In [ ]:
ensure_notebook_requirements('notebook02')

import json
import random
from pathlib import Path

from src.dataset import format_for_training, load_stepgame

## Download StepGame

StepGame is available on HuggingFace datasets or from the original repo.
Run the cell below once to download raw splits.

In [ ]:
from datasets import load_dataset

ds = load_dataset("ZhengyanShi/StepGame")
print(ds)


In [ ]:
# Inspect a sample
print(ds['train'][0])

## Adapt field names if needed

StepGame fields vary by source. Adjust `story_field`, `question_field`, `answer_field` below.

In [ ]:
# ZhengyanShi/StepGame field mapping
STORY_FIELD    = "story"
QUESTION_FIELD = "question"
ANSWER_FIELD   = "label"
K_FIELD        = "k_hop"

TRAIN_SIZE     = 10000  # 2000 per k level
TRAIN_PER_K    = 2000
EVAL_PER_K     = 50   # 50 examples per hop level → 250 total across k=1..5
SEED           = 42

# 8 valid directions — "overlap" is not a spatial direction, filter it out
VALID_ANSWERS  = {"left", "right", "above", "below", "upper-left", "upper-right", "lower-left", "lower-right"}

def convert(ex):
    return {
        "story":    " ".join(ex[STORY_FIELD]) if isinstance(ex[STORY_FIELD], list) else ex[STORY_FIELD],
        "question": ex[QUESTION_FIELD],
        "answer":   ex[ANSWER_FIELD],
        "k":        ex[K_FIELD],
    }

from collections import defaultdict
import random
random.seed(SEED)

# Train: stratified sample — TRAIN_PER_K examples per k level
by_k_train = defaultdict(list)
for ex in ds["train"]:
    if ex[ANSWER_FIELD].strip().lower() in VALID_ANSWERS:
        by_k_train[ex[K_FIELD]].append(ex)

train_data = []
for k in sorted(by_k_train):
    sample = random.sample(by_k_train[k], min(TRAIN_PER_K, len(by_k_train[k])))
    train_data.extend(convert(ex) for ex in sample)

random.shuffle(train_data)

# Eval: stratified sample — EVAL_PER_K examples per k level, no overlap answers
by_k_eval = defaultdict(list)
for ex in ds["validation"]:
    if ex[ANSWER_FIELD].strip().lower() in VALID_ANSWERS:
        by_k_eval[ex[K_FIELD]].append(ex)

eval_data = []
for k in sorted(by_k_eval):
    sample = random.sample(by_k_eval[k], min(EVAL_PER_K, len(by_k_eval[k])))
    eval_data.extend(convert(ex) for ex in sample)

print(f"Train: {len(train_data)}")
for k in sorted(by_k_train):
    n = sum(1 for ex in train_data if ex["k"] == k)
    print(f"  k={k}: {n} examples")
print(f"Eval:  {len(eval_data)}")
for k in sorted(by_k_eval):
    n = sum(1 for ex in eval_data if ex["k"] == k)
    print(f"  k={k}: {n} examples")

In [ ]:
# Format for fine-tuning
train_formatted = [format_for_training(ex) for ex in train_data]

# Save
raw_dir = PATHS['data_root'] / 'raw'
processed_dir = PATHS['data_root'] / 'processed'
eval_dir = PATHS['data_root'] / 'eval'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
eval_dir.mkdir(parents=True, exist_ok=True)

with open(raw_dir / 'train.json', 'w') as f:
    json.dump(train_data, f, indent=2)

with open(processed_dir / 'train_formatted.json', 'w') as f:
    json.dump(train_formatted, f, indent=2)

with open(eval_dir / 'stepgame_eval.json', 'w') as f:
    json.dump(eval_data, f, indent=2)

print('Saved.')
print('Sample formatted:')
print(train_formatted[0]['full_text'][:500])


In [ ]:
publish_artifacts(
    [
        'data/raw/train.json',
        'data/processed/train_formatted.json',
        'data/eval/stepgame_eval.json',
    ],
    'Refresh StepGame training and eval data [notebook 02]',
    repo_dir=REPO,
)
